# Ein Enzym bei der Arbeit — und die 81 Atome, um die es wirklich geht

Wir schauen uns eine **klassische Molekulardynamik-Simulation** an: eine Pseudouridin-Synthase
im Komplex mit ihrer RNA, in Wasser, mit Ionen — 88 311 Atome, 20 gespeicherte Zeitpunkte.

Der Weg durch das Notebook ist ein Zoom:

| Schritt | Was wir sehen | Größenordnung |
|---|---|---|
| 1–4 | das ganze System, inventarisiert | ~88 000 Atome |
| 5–7 | Ketten und QM-Region, geprüft | 81 Atome |
| 8–9 | Enzym + RNA, markiert und in Bewegung | ~4 300 Atome, 20 Frames |
| 10 | alles zurück, inklusive Lösungsmittel | 88 311 Atome |

Die Pointe für die Sitzung steht am Ende — sie ergibt sich aus dem Verhältnis der beiden
Zahlen, die wir unterwegs ausrechnen.

> Jede Zelle macht genau eine Sache und lässt sich einzeln lesen. Von oben nach unten
> durchlaufen lassen (in Colab: `Strg`/`Cmd` + `F9`).

## 1 · Werkzeuge

Nur zwei Pakete. `mdtraj` liest Topologie und Trajektorie, `py3Dmol` zeichnet sie im Browser —
kein Widget-Manager, kein JavaScript-Setup, nichts, was in Colab kaputtgehen kann.

In [ ]:
!pip install -q py3Dmol mdtraj

## 2 · Dateien

Zwei Dateien, wie bei jeder MD-Simulation:

- **`topology.pdb`** — *wer* ist da: Atome, Elemente, Reste, Bindungen. Einmalig, unveränderlich.
- **`reduced_traj.dcd`** — *wo* sie sind: nur Koordinaten, ein Satz pro Frame. Deshalb ist die
  Trajektorie ein reines Zahlenfeld und braucht die Topologie, um interpretierbar zu sein.

In [ ]:
import os, urllib.request

REPO = "https://raw.githubusercontent.com/robert-scr/QCforQC/main"

for name in ["topology.pdb", "reduced_traj.dcd"]:
    if not os.path.exists(name):                       # schon da? dann nicht nochmal laden
        urllib.request.urlretrieve(f"{REPO}/{name}", name)
    print(f"{name:<18} {os.path.getsize(name)/1e6:6.1f} MB")

## 3 · Laden

`md.load` legt beides zusammen: Koordinaten aus der `.dcd`, Bedeutung aus der `.pdb`.
Heraus kommt ein Array der Form `(Frames, Atome, 3)` — in **Nanometern**, nicht in Ångström.

In [ ]:
import mdtraj as md
import numpy as np

traj = md.load("reduced_traj.dcd", top="topology.pdb")
topo = traj.topology


def zahl(n):
    """1234567 -> '1 234 567' (deutsche Schreibweise)."""
    return f"{n:,}".replace(",", " ")


print(traj, "\n")
print(f"Frames        : {traj.n_frames}")
print(f"Atome         : {zahl(traj.n_atoms)}")
print(f"Reste         : {zahl(topo.n_residues)}")
print(f"Koordinaten   : {traj.xyz.shape}  (Frames, Atome, xyz) in nm")
print(f"Simulationsbox: {np.round(traj.unitcell_lengths[0], 2)} nm")

## 4 · Inventur — woraus besteht das System?

Bevor irgendetwas gezeichnet wird: **zählen**. Die Prozentspalte ist schon die halbe Botschaft
dieses Notebooks.

In [ ]:
RNA_RESTE = {"A", "U", "G", "C", "A3", "U3", "G5"}     # 3'- und 5'-Enden tragen eigene Namen
ION_RESTE = {"K+", "Cl-", "ZN"}

gruppen = {
    "Protein (Enzymkomplex)": [r for r in topo.residues if r.is_protein],
    "RNA":                    [r for r in topo.residues if r.name in RNA_RESTE],
    "Ionen (K+, Cl-, Zn)":    [r for r in topo.residues if r.name in ION_RESTE],
    "Wasser":                 [r for r in topo.residues if r.name == "HOH"],
}

for name, reste in gruppen.items():
    n_atome = sum(r.n_atoms for r in reste)
    print(f"{name:<24} {zahl(len(reste)):>7} Reste  {zahl(n_atome):>8} Atome   {100*n_atome/traj.n_atoms:5.1f} %")

88 % der Atome sind Wasser. Das ist kein Ballast, sondern der Grund, warum die Simulation
überhaupt etwas über ein Enzym in einer Zelle aussagt — und zugleich der Grund, warum diese Art
von Rechnung mit Quantenmechanik nichts zu tun hat: Wasser wird hier als Kugeln an Federn
behandelt, nicht als Elektronensystem.

## 5 · Ketten sichtbar machen

Die PDB-Datei kommt aus einem MD-Programm und steckt **alle** Atome in eine einzige Kette `A`.
Für die Cartoon-Darstellung ist das fatal: der Zeichner verbindet dann Reste, die gar nicht
verbunden sind, und legt ein Band quer durch die Box.

Also vergeben wir die Ketten selbst — anhand der Restnummern, die wir aus der Inventur kennen.

In [ ]:
KETTEN = {
    "A": "protein and resid   0 to 326",   # große Untereinheit, trägt das aktive Zentrum
    "B": "protein and resid 327 to 379",   # kleine Untereinheit
    "C": "protein and resid 380 to 499",   # weitere Untereinheit
    "D": "resid 500 to 557",               # RNA-Strang 1
    "E": "resid 558 to 570",               # RNA-Strang 2 — enthält das Ziel-Uridin
    "I": "resname 'K+' or resname 'Cl-' or resname ZN",
    "W": "water",
}

ketten_id = np.full(traj.n_atoms, "X", dtype="<U1")    # X = noch nicht zugeordnet
for kette, auswahl in KETTEN.items():
    treffer = topo.select(auswahl)
    ketten_id[treffer] = kette
    print(f"Kette {kette}: {zahl(len(treffer)):>8} Atome   ({auswahl})")

assert not (ketten_id == "X").any(), "es sind Atome übrig geblieben"

## 6 · Die QM-Region

In einer QM/MM-Rechnung wird genau ein kleiner Ausschnitt quantenmechanisch behandelt; der
gesamte Rest bleibt klassisches Kraftfeld. Dieser Ausschnitt ist hier vorgegeben — er stammt
aus der zugrundeliegenden Rechnung und ist als Liste von Atomindizes definiert.

Was drin ist: das **Substrat-Uridin** mit den beiden benachbarten Nukleotid-Anschlüssen, die
**Aspartat-Seitenkette** (der in Pseudouridin-Synthasen konservierte katalytische Rest), zwei
weitere Seitenketten der unmittelbaren Umgebung und ein **Kalium-Ion**.

Wichtig für später: die Region ist an Bindungen aufgeschnitten. In der echten Rechnung werden
die offenen Enden mit Link-Atomen abgesättigt — eine QM-Region ist ein *Ausschnitt*, kein Molekül.

In [ ]:
QM_GRUPPEN = {
    "Asp — katalytische Seitenkette": range(1191, 1197),
    "Tyr — Seitenkette":              range(1617, 1632),
    "Arg — Seitenkette":              range(3139, 3160),
    "Substrat-RNA (U + Nachbarn)":    range(10109, 10147),
    "Kalium-Ion":                     range(10364, 10365),
}

qm = sorted({i for gruppe in QM_GRUPPEN.values() for i in gruppe})

for name, gruppe in QM_GRUPPEN.items():
    reste = sorted({f"{topo.atom(i).residue.name}{topo.atom(i).residue.resSeq}" for i in gruppe})
    print(f"{name:<32} {len(list(gruppe)):>3} Atome   {', '.join(reste)}")

print(f"\nQM-Region gesamt: {len(qm)} Atome")

### Probe: hängt die Region überhaupt zusammen?

Atomindizes sind eine gefährliche Währung — sie gelten nur für exakt diese Topologie, und ein
falscher Bereich fällt beim bloßen Hinsehen nicht auf. Deshalb eine billige Kontrolle:
eine QM-Region muss **räumlich kompakt** sein. Ein paar Ångström, nicht ein paar Nanometer.

In [ ]:
qm_xyz    = traj.xyz[0][qm] * 10.0                     # nm -> Ångström
qm_mitte  = qm_xyz.mean(axis=0)
abstaende = np.linalg.norm(qm_xyz - qm_mitte, axis=1)
qm_radius = float(abstaende.max()) + 2.0               # +2 Å Luft für die Markierungskugel

for name, gruppe in QM_GRUPPEN.items():
    d = np.linalg.norm(traj.xyz[0][list(gruppe)] * 10.0 - qm_mitte, axis=1).mean()
    print(f"{name:<32} Schwerpunkt {d:5.1f} Å von der Mitte")

print(f"\nweitestes Atom: {abstaende.max():.1f} Å — die Region ist kompakt.")

### Und das Wasser?

Eine echte QM/MM-Rechnung nimmt meist noch die Wassermoleküle im aktiven Zentrum dazu. Nur:
*welche*? Wasser tauscht aus. Über einen festen Atomindex ist ein Lösungsmittelmolekül nicht zu
fassen — man muss in jedem Frame neu suchen.

In [ ]:
wasser_o = topo.select("water and name O")

for frame in [0, traj.n_frames // 2, traj.n_frames - 1]:
    nah = md.compute_neighbors(traj[frame], 0.35, qm, haystack_indices=wasser_o)[0]
    reste = sorted(topo.atom(i).residue.resSeq for i in nah)
    print(f"Frame {frame:>2}: {len(nah):>2} Wasser innerhalb 3,5 Å   {reste}")

Ein paar Wasser bleiben über alle Frames liegen, andere kommen und gehen. Wo genau die QM-Region
aufhört, ist damit **eine Entscheidung und keine Naturkonstante** — und sie beeinflusst das
Ergebnis. Wir lassen das Wasser hier draußen und rechnen mit den 81 Atomen oben weiter.

Und jetzt die Zahl, auf die es ankommt — die Elektronen. Für ein Atom mit Kernladungszahl $Z$
sind es im neutralen Fall $Z$ Elektronen; die Summe über die Region ist also die
Größenordnung, mit der eine Elektronenstrukturrechnung hier zu tun hätte.

In [ ]:
n_elektronen = sum(topo.atom(i).element.number for i in qm)
n_schwer     = sum(1 for i in qm if topo.atom(i).element.symbol != "H")

print(f"QM-Region  : {len(qm)} Atome  ({n_schwer} davon Nicht-Wasserstoff)")
print(f"Elektronen : {n_elektronen}   (Summe der Kernladungszahlen, neutral gerechnet —")
print(f"             die echte Zahl weicht um die Nettoladung der Region ab)")
print()
print(f"Anteil am simulierten System: {len(qm)} / {zahl(traj.n_atoms)}"
      f" = {100*len(qm)/traj.n_atoms:.3f} %")

## 7 · Ein PDB schreiben, das py3Dmol versteht

Eine kleine Hilfsfunktion, und danach ist Schluss mit Dateiformat-Details. Sie tut drei Dinge:

1. schneidet die gewünschten Atome heraus und schreibt sie als PDB,
2. trägt unsere **Ketten-IDs** in Spalte 22 ein,
3. benennt Randfälle um, die der Zeichner sonst nicht als Standardreste erkennt
   (`G5` → `G`, `U3` → `U`, `CYM` → `CYS`) und wirft die `CONECT`-Zeilen weg, deren
   Nummerierung beim Herausschneiden nicht mehr stimmt.

Praktisch dabei: `mdtraj` behält beim Herausschneiden die **ursprüngliche Seriennummer** jedes
Atoms. Ein Atom hat also in jeder Ansicht dieselbe Nummer, und wir können die QM-Region später
einfach über ihre Seriennummern ansprechen.

In [ ]:
RESTNAME_FIX = {"G5": "G", "A3": "A", "U3": "U", "CYM": "CYS"}


def schreibe_pdb(t, indizes, pfad):
    """Schreibt ausgewählte Atome von t als PDB und gibt den Text zurück."""
    t.atom_slice(np.sort(np.asarray(indizes))).save_pdb(pfad)

    zeilen = []
    for z in open(pfad):
        if z.startswith(("ATOM  ", "HETATM")):
            index = int(z[6:11]) - 1                                  # Seriennummer -> Atomindex
            rest  = RESTNAME_FIX.get(z[17:20].strip(), z[17:20].strip())
            zeilen.append(f"{z[:17]}{rest:>3} {ketten_id[index]}{z[22:]}")
        elif z.startswith(("CRYST1", "MODEL", "ENDMDL", "END")):      # Rahmen behalten
            zeilen.append(z)

    text = "".join(zeilen)
    open(pfad, "w").write(text)
    return text


# Die Seriennummern der QM-Atome — unsere Handhabe für alle folgenden Darstellungen.
qm_serials = [int(i) + 1 for i in qm]
print("QM-Seriennummern:", qm_serials[:8], "...", qm_serials[-3:])

## 8 · Standbild: Enzym, RNA, QM-Region

Für die Darstellung reicht ein Bruchteil der Atome:

- vom Protein nur das **Rückgrat** (`N`, `CA`, `C`, `O`) — mehr braucht ein Cartoon nicht,
- die **RNA** vollständig,
- die **QM-Region** vollständig.

Aus 88 311 Atomen werden so gut 4 000 — der Browser dankt es.

In [ ]:
import py3Dmol

rueckgrat = topo.select("protein and (name N or name CA or name C or name O)")
rna       = topo.select(" or ".join(f"resname '{n}'" for n in sorted(RNA_RESTE)))
ANSICHT   = np.unique(np.concatenate([rueckgrat, rna, np.asarray(qm)]))

print(f"Darstellung: {zahl(len(ANSICHT))} von {zahl(traj.n_atoms)} Atomen")


def punkt(v):
    return {"x": float(v[0]), "y": float(v[1]), "z": float(v[2])}


def stil_setzen(view, markieren=True):
    """Einheitliche Darstellung: Protein grau, RNA gold, QM-Region cyan."""
    view.setStyle({}, {})                                            # erst alles ausblenden
    view.setStyle({"chain": ["A", "B", "C"]},
                  {"cartoon": {"color": "lightgrey", "opacity": 0.9}})
    view.setStyle({"chain": ["D", "E"]},
                  {"cartoon": {"color": "goldenrod"},
                   "stick":   {"radius": 0.07, "color": "goldenrod"}})
    # QM-Region zuletzt und mit addStyle: sie liegt *auf* dem Cartoon, statt es zu ersetzen
    view.addStyle({"serial": qm_serials},
                  {"stick":  {"radius": 0.16, "colorscheme": "cyanCarbon"},
                   "sphere": {"scale": 0.28,  "colorscheme": "cyanCarbon"}})
    if markieren:
        view.addSphere({"center": punkt(qm_mitte), "radius": qm_radius,
                        "color": "cyan", "opacity": 0.20})
        view.addLabel(f"QM-Region · {len(qm)} Atome",
                      {"position": punkt(qm_mitte + [0, 0, qm_radius + 2]),
                       "backgroundColor": "black", "backgroundOpacity": 0.6,
                       "fontColor": "white", "fontSize": 13})

In [ ]:
text = schreibe_pdb(traj[0], ANSICHT, "ansicht_frame0.pdb")

view = py3Dmol.view(width=900, height=620)
view.addModel(text, "pdb")
stil_setzen(view)

view.zoomTo({"serial": qm_serials})   # auf die QM-Region zielen ...
view.zoom(0.30)                       # ... und wieder heraus, bis das Enzym drumherum im Bild ist
view.show()

Zum Selbermachen: `view.zoom(...)` ändern (größer = näher dran), oder die letzten beiden Zeilen
durch `view.zoomTo()` ersetzen, um den ganzen Komplex zu sehen. Mit der Maus drehen, mit dem
Rad zoomen, mit gedrückter mittlerer Taste verschieben.

## 9 · Die Trajektorie in Bewegung

Ein Punkt vorweg, der oft übersehen wird: die Simulationsbox darf sich als Ganzes drehen und
verschieben. Bevor man Bewegung *im* Molekül sehen will, muss man diese Gesamtbewegung
herausrechnen — hier durch Überlagerung aller Frames auf Frame 0 anhand der Cα-Atome.

py3Dmol animiert dann einfach eine PDB-Datei mit einem `MODEL`-Block pro Frame.

In [ ]:
ca = topo.select("protein and name CA")

traj_fit = traj[:]                                     # Kopie — die Originaldaten bleiben heil
traj_fit.superpose(traj_fit, frame=0, atom_indices=ca)

text = schreibe_pdb(traj_fit, ANSICHT, "ansicht_traj.pdb")
print(f"{traj.n_frames} Frames × {zahl(len(ANSICHT))} Atome  →  {len(text)/1e6:.1f} MB PDB-Text")

view = py3Dmol.view(width=900, height=620)
view.addModelsAsFrames(text, "pdb")                    # ein MODEL-Block = ein Frame
stil_setzen(view, markieren=True)

view.zoomTo()
view.animate({"loop": "forward", "interval": 250})     # interval = ms pro Frame
view.show()

Was man sieht: das Enzym atmet, die RNA-Schleifen zappeln, die cyanfarbene QM-Region wackelt an
ihrem Platz. Was man **nicht** sieht: eine Reaktion. Ein klassisches Kraftfeld kann Bindungen
weder brechen noch knüpfen — es kennt keine Elektronen, nur Federkonstanten. Genau an dieser
Grenze setzt QM/MM an: klassisch überall, quantenmechanisch nur in der markierten Kugel.

*(20 Frames sind eine stark ausgedünnte Trajektorie — genug, um Bewegung zu sehen, klein genug
für ein Notebook. Die zugrundeliegende Simulation hat entsprechend mehr Schritte.)*

## 10 · Und jetzt alles

Ein einzelner Frame, aber vollständig: Enzym, RNA, Ionen, Lösungsmittel, periodische Box.
So sieht das aus, was tatsächlich integriert wird.

Das Wasser wird nur über seine Sauerstoffatome gezeigt (25 891 Punkte statt 77 673) —
sonst wird das Bild zu einer undurchsichtigen Wand. `MIT_WASSERSTOFF = True` setzen,
wenn man genau das sehen will.

In [ ]:
MIT_WASSERSTOFF = False

wasser = topo.select("water") if MIT_WASSERSTOFF else topo.select("water and name O")
alles  = np.unique(np.concatenate([topo.select("not water"), wasser]))
print(f"Dargestellt: {zahl(len(alles))} Atome")

text = schreibe_pdb(traj[0], alles, "alles_frame0.pdb")


def zeichne_box(view, laengen, farbe="grey"):
    """Kanten der orthorhombischen Simulationsbox (Ursprung bei 0)."""
    lx, ly, lz = (float(v) for v in laengen)
    ecken = [(x, y, z) for x in (0, lx) for y in (0, ly) for z in (0, lz)]
    for a in ecken:
        for b in ecken:
            if a < b and sum(a[k] != b[k] for k in range(3)) == 1:      # genau eine Kante
                view.addLine({"start": punkt(a), "end": punkt(b), "color": farbe})


view = py3Dmol.view(width=900, height=700)
view.addModel(text, "pdb")

view.setStyle({}, {})
view.setStyle({"chain": "W"}, {"sphere": {"radius": 0.32, "color": "lightblue", "opacity": 0.55}})
view.setStyle({"chain": "I"}, {"sphere": {"scale": 0.45}})
view.setStyle({"chain": ["A", "B", "C"]}, {"cartoon": {"color": "lightgrey", "opacity": 0.95}})
view.setStyle({"chain": ["D", "E"]}, {"cartoon": {"color": "goldenrod"},
                                      "stick":   {"radius": 0.08, "color": "goldenrod"}})
view.addStyle({"serial": qm_serials}, {"stick":  {"radius": 0.16, "colorscheme": "cyanCarbon"},
                                       "sphere": {"scale": 0.28,  "colorscheme": "cyanCarbon"}})

zeichne_box(view, traj.unitcell_lengths[0] * 10.0)     # nm -> Ångström

view.zoomTo()
view.show()

## Was das mit Quantencomputern zu tun hat

Drei Zahlen aus diesem Notebook nebeneinander:

| | |
|---|---|
| simuliertes System | **88 311 Atome** |
| davon Lösungsmittel | **88 %** |
| QM-Region | **81 Atome — 0,09 %** |

Und daraus drei Aussagen, die sauber auseinandergehalten werden müssen:

**Erstens.** Was hier animiert wurde — Konformationsdynamik, Lösungsmittel, Bewegung über
Nanosekunden — ist klassische Statistik über sehr viele Zustände. Ein Quantencomputer macht das
**nicht**. Nicht „noch nicht", sondern: das ist nicht die Aufgabe. Wer Proteindynamik,
Bindungsaffinitäten oder Docking als Anwendung von Quantencomputing verkauft bekommt, sollte
genau an dieser Stelle nachfragen.

**Zweitens.** Elektronen — und damit überhaupt erst Chemie im Sinne von Bindungsbruch und
-knüpfung — kommen nur in der cyanfarbenen Kugel vor. Das ist der einzige Teil des Bildes, für
den die Frage nach einem Quantencomputer überhaupt sinnvoll gestellt ist. Er ist ein
Tausendstel des Systems, und er ist bereits ein *Ausschnitt*, aufgeschnitten und mit
Link-Atomen abgesättigt.

**Drittens.** Auch dort lautet die Frage nicht „Quantencomputer ja oder nein", sondern: reicht
DFT? Reicht CCSD(T)? Fast immer lautet die Antwort ja — und dann ist das Problem gelöst, ohne
dass irgendjemand ein Qubit angefasst hat. Der schmale Rest, bei dem das nicht reicht, ist genau
das Thema der Sitzung.

---

**Denkaufgabe.** Die QM-Region hat knapp **400 Elektronen**. Eine Full-CI-Rechnung würde alle
Möglichkeiten aufzählen, diese Elektronen auf die Orbitale einer Basis zu verteilen. Wie viele
Determinanten wären das? Bevor gerechnet wird: schätzen. Dann den Determinanten-Zähler aus dem
Vortrag anwerfen — und beobachten, wie weit die Schätzung danebenlag.